In [10]:
import numpy as np
import string 

sanitizer = lambda x: x.lower().translate(str.maketrans('', '', string.punctuation)).strip().replace("—", " ") #patchwork ass sanitization function that removes leading spaces, puncutation and replaces - with a space
clean_script_list = np.loadtxt("../data/hobbit1.csv", delimiter=',', usecols = 1,  skiprows = 1, dtype=str, converters = sanitizer, encoding="utf-8")

tokenized_script = [sentence.split() for sentence in clean_script_list] #split into 2d array of sentences split into words

vocab = sorted(set(word for sentence in tokenized_script for word in sentence)) #get a sorted set of unique words
word_to_index = {word: idx for idx, word in enumerate(vocab)} #initialize a dict, word corresponds to a value index

def one_hot_generator(word, vocab_size): 
    one_hot = np.zeros(vocab_size)
    one_hot[word_to_index[word]] = 1 
    return one_hot


In [11]:
#need a user specified window-size, hardcode this to 3 for now 
#CBOW IMPLEMENTATION
WINDOW_SIZE = 3

def CBOW_generate_training_data(tokenized_sen, word_dict, window_size = 1):
    X_train = []
    y_train = []

    V = len(word_dict)

    for sen in tokenized_sen: #at this point, we assume our data is a 2d array, of tokenized sentences, so iterate through those sen
        for i, target in enumerate(sen): #slide through whole sentece, each word is a target word, no loop backs here for now

            aggregated_cv = np.zeros(V)

            
            start = max(0, i - window_size) #the start is capped to the first 
            end = min(len(sen), i + window_size) #end is capped to last element

            context = [neighbor for neighbor in sen[start:end + 1] if neighbor != target] #comprehension list, iterate through all words in our sublist window, keep target word out.
            for word in context: 
                aggregated_cv[word_dict[word]] += 1.0 

            if len(context) > 0:
                aggregated_cv /= len(context)

            y_train.append(one_hot_generator(target, V)) #and append to context list to data matrix, and corresponding one hot target word rep in target list.
            X_train.append(aggregated_cv)

    return np.array(X_train).T, np.array(y_train).T
#np.reshape(np.array(X_train), (-1, len(vocab))), np.reshape(np.array(y_train), (-1, 1))

X_train, y_train = CBOW_generate_training_data(tokenized_script, word_to_index, 2)


In [ ]:
def softmax(X):
    X_max = np.max(X, axis=0, keepdims=True)
    e_X = np.exp(X - X_max)
    #something about numerical stability will add later if i understand it 
    return e_X / np.sum(e_X, axis = 0, keepdims = True) #dumb luck

def forward_prop(X, W0, W1, b0, b1):

    #X : aggregated onehot input for group of words
    A1 = W0.T @ X + b0 #U: intermediate output, context vectors
    #no activation function on hidden layer output  
    A2 = W1.T @ A1 + b1 #Prediction between vectors 
    Y_predict= softmax(A2) #Y: apply softmax to obtain final prediction, will be plugged into loss
    return A1, A2, Y_predict

def update_model(W0, W1, b0, b1, dLdW0, dLdW1, dLdb0, dLdb1, alpha):
    W1_new = W1 - alpha * dLdW1
    W0_new = W0 - alpha * dLdW0

    b1_new = b1 - alpha * dLdb1 
    b0_new = b0 - alpha * dLdb0
    return W0_new, W1_new, b0_new, b1_new

def print_summary(Y, Y_predict, iteration): 
    loss = -np.mean(np.sum(Y * np.log(Y_predict + 1e-12), axis=0))
    true_val = np.argmax(Y, axis = 0)
    predict_val = np.argmax(Y_predict, axis = 0)
    accuracy = np.mean(true_val == predict_val)
    print(f'iteration: {iteration}')
    print(f"loss: {loss}")
    print(f"accuracy: {accuracy}")

def grad_desc(X, Y, W0, W1, b0, b1, alpha = 0.01, epochs = 50):
    M = X.shape[1]
    for i in range(epochs): 
        A1, A2, Y_predict = forward_prop(X, W0, W1, b0, b1)
        
        dLdb1, dLdb0 = 0, 0
        dLdu = Y_predict - Y # partial deriv dL/du
        dLdW1 = (1 / M) * (A1 @ dLdu.T)
        dLdb1 = (1 / M) * (dLdu @ np.ones((M, 1)))
        dLdW0 = (1 / M) * (X @ (W1 @ dLdu).T)
        
        W0, W1, b0, b1 = update_model(W0, W1, b0, b1, dLdW0, dLdW1, dLdb0, dLdb1, alpha)

        if(i % 5 == 0):
            print_summary(Y, Y_predict, i)
    return W0, W1, b0, b1

def stoch_grad_desc(X, Y, W0, W1, b0, b1, alpha = 0.01, epochs = 50):
    M = X.shape[1]
    for i in range(epochs): 
        for j in range(M): 
            A1, A2, Y_predict = forward_prop(X[:,[j]], W0, W1, b0, b1)

            dLdb1, dLdb0 = 0, 0
            dLdu = Y_predict - Y[:,[j]] # partial deriv dL/du
            dLdW1 = (A1 @ dLdu.T)
            dLdb1 = dLdu
            dLdW0 = (X[:,[j]] @ (W1 @ dLdu).T)
            
            W0, W1, b0, b1 = update_model(W0, W1, b0, b1, dLdW0, dLdW1, dLdb0, dLdb1, alpha)

            if(j % 5 == 0):
                print_summary(Y, Y_predict, j)

        
    return W0, W1, b0, b1



EMBEDDING_DIM = int(np.sqrt(len(vocab))) #hard coded for now

def init_vals(): 
    W0 = np.random.randn(len(vocab), EMBEDDING_DIM) #matrix to map to trait nodes
    W1 = np.random.randn(EMBEDDING_DIM, len(vocab)) #
    b0 = 0.0
    b1 = 0.0
    return W0, W1, b0, b1 

#for now we'll stick with vanilla gradient descent, will figure out stochastic later.

W0, W1, b0, b1 = init_vals()
W0, W1, b0, b1  = grad_desc(X_train, y_train, W0, W1, b0, b1, 10.0, 100)


while(True): 
    max_context_words = 3
    context_words = []
    for i in range(max_context_words):
        word = ''
        while(word not in word_to_index): 
            word = input()
            if(word in word_to_index): 
                break
            print("not in vocab")
        
        print(f'added word: {word}')
        context_words.append(word)

    aggregated_cv = np.zeros((len(vocab), 1))

    for word in context_words: 
        aggregated_cv[word_to_index[word]] += 1.0 

    A0, A1, prediction = forward_prop(aggregated_cv, W0, W1, b0, b1)
    print(prediction.shape)
    print(vocab[np.argmax(prediction.flatten(), axis = 0)])

    


    

iteration: 0
loss: 12.832526693439357
accuracy: 0.0010384215991692627
iteration: 5
loss: 11.04365502987257
accuracy: 0.003115264797507788
iteration: 10
loss: 9.897942783955202
accuracy: 0.007268951194184839
iteration: 15
loss: 9.086674481973839
accuracy: 0.015368639667705089
iteration: 20
loss: 8.482892414965498
accuracy: 0.02679127725856698
iteration: 25
loss: 8.011839581506443
accuracy: 0.03613707165109034
iteration: 30
loss: 7.627458060725552
accuracy: 0.04340602284527518
iteration: 35
loss: 7.303817086867182
accuracy: 0.05254413291796469
iteration: 40
loss: 7.02586967698166
accuracy: 0.06396677050882658
iteration: 45
loss: 6.7830182644522
accuracy: 0.07123572170301143
iteration: 50
loss: 6.567940878470443
accuracy: 0.08016614745586709
iteration: 55
loss: 6.3753766473937175
accuracy: 0.08868120456905504
iteration: 60
loss: 6.201017434614185
accuracy: 0.09740394600207684
iteration: 65
loss: 6.041484148948681
accuracy: 0.1059190031152648
iteration: 70
loss: 5.894328212622095
accuracy:

AttributeError: module 'numpy' has no attribute 'flatten'